In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation, LeakyReLU, UpSampling2D, Conv2D
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam

import numpy as np
import os
from PIL import Image


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
def load_images(path, image_size):
    image_files = os.listdir(path)
    images = []
    for filename in image_files:
        try:
            img = Image.open(os.path.join(path, filename)).resize((image_size, image_size))
            img = np.array(img)
            if img.shape == (image_size, image_size, 3):  # Ensure images are RGB
                images.append(img)
        except Exception as e:
            print(f"Skipping {filename}: {e}")
    return np.array(images) / 255.0  # Normalize pixel values to [0, 1]

# Example usage
image_size = 1024  # Adjust according to your dataset
images = load_images('/content/gdrive/MyDrive/new/', image_size)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, UpSampling2D, Conv2D, BatchNormalization, Activation
import tensorflow as tf

def build_generator(latent_dim):
    model = Sequential()

    # Determine initial dimensions after reshaping from latent space
    initial_width = 32
    initial_height = 32
    initial_channels = 256

    model.add(Dense(initial_width * initial_height * initial_channels, activation="relu", input_dim=latent_dim))
    model.add(Reshape((initial_width, initial_height, initial_channels)))

    # Upsample to 64x64
    model.add(UpSampling2D())
    model.add(Conv2D(256, kernel_size=3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Upsample to 128x128
    model.add(UpSampling2D())
    model.add(Conv2D(128, kernel_size=3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Upsample to 256x256
    model.add(UpSampling2D())
    model.add(Conv2D(64, kernel_size=3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Upsample to 512x512
    model.add(UpSampling2D())
    model.add(Conv2D(32, kernel_size=3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Upsample to 1024x1024
    model.add(UpSampling2D())
    model.add(Conv2D(3, kernel_size=3, padding="same", activation="tanh"))

    return model


# Image dimensions
image_size = 256
image_shape = (image_size, image_size, 3)

# Latent space dimension
latent_dim = 100

# Build the generator
generator = build_generator(latent_dim)




def build_discriminator(image_shape):
    model = Sequential()

    model.add(Conv2D(64, kernel_size=3, strides=2, input_shape=image_shape, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(128, kernel_size=3, strides=2, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(256, kernel_size=3, strides=2, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(512, kernel_size=3, strides=2, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(1024, kernel_size=3, strides=2, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))

    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))

    return model



# Image dimensions
image_size = 1024
image_shape = (image_size, image_size, 3)
latent_dim = 100

# Build and compile the discriminator
discriminator = build_discriminator(image_shape)
discriminator.compile(loss='binary_crossentropy',
                      optimizer=Adam(0.0002, 0.5),
                      metrics=['accuracy'])

# Build the generator
generator = build_generator(latent_dim)

# Generate image from random noise
z = Input(shape=(latent_dim,))
img = generator(z)

# For the combined model we will only train the generator
discriminator.trainable = False

# The discriminator takes generated images as input and determines validity
validity = discriminator(img)

# The combined model (stacked generator and discriminator)
# Trains the generator to fool the discriminator
combined = Model(z, validity)
combined.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))



In [ ]:
def train_gan(generator, discriminator, combined, images, epochs, batch_size=1, save_interval=5):
    # Rescale -1 to 1
    X_train = images * 2 - 1

    # Adversarial ground truths
    valid = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    for epoch in range(epochs):

        # ---------------------
        #  Train Discriminator
        # ---------------------

        # Select a random half of images
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        imgs = X_train[idx]

        # Sample noise and generate a batch of new images
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        gen_imgs = generator.predict(noise)

        # Train the discriminator (real classified as ones and generated as zeros)
        d_loss_real = discriminator.train_on_batch(imgs, valid)
        d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # ---------------------
        #  Train Generator
        # ---------------------

        # Train the generator (wants discriminator to mistake images as real)
        g_loss = combined.train_on_batch(noise, valid)

        # Plot the progress
        print(f"{epoch} [D loss: {d_loss[0]}, acc.: {100 * d_loss[1]}] [G loss: {g_loss}]")

        # If at save interval => save generated image samples
        if epoch % save_interval == 0:
            save_generated_images(generator, epoch)

def save_generated_images(generator, epoch, examples=10, latent_dim=100):
    noise = np.random.normal(0, 1, (examples, latent_dim))
    generated_images = generator.predict(noise)

    # Rescale images 0 - 1
    generated_images = 0.5 * generated_images + 0.5

    for i in range(examples):
        img = Image.fromarray((generated_images[i] * 255).astype(np.uint8))
        img.save(f"generated_image_{epoch}_{i}.png")

# Set hyperparameters and train the GAN
epochs = 500
batch_size = 4
train_gan(generator, discriminator, combined, images, epochs, batch_size)


In [ ]:
def generate_images(generator, latent_dim, examples=10):
    noise = np.random.normal(0, 1, (examples, latent_dim))
    generated_images = generator.predict(noise)

    # Rescale images 0 - 1
    generated_images = 0.5 * generated_images + 0.5

    for i in range(examples):
        img = Image.fromarray((generated_images[i] * 255).astype(np.uint8))
        img.show()  # Display or save the image as needed

# Example of generating images
generate_images(generator, latent_dim)
